# 🔄 ETL Bronze → Silver (Spark Job)
## Crime Data Pipeline

Job de transformação para execução via Spark.

**Objetivo**: Transformar dados da camada Bronze para Silver com limpeza, validação e feature engineering.

In [ ]:
# Configurações do Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Iniciar sessão Spark
spark = SparkSession.builder \
    .appName("BronzeToSilver_CrimeData") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print(f"✅ Spark Session iniciada: {spark.version}")

In [ ]:
# Configuração de caminhos (detectar raiz do projeto)
import os
from pathlib import Path

def find_project_root() -> Path:
    """Encontra a raiz do projeto SBD2"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for root in candidates:
        if (root / 'Crime_Data_from_2020_to_Present.csv').exists():
            return root
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
BRONZE_PATH = str(PROJECT_ROOT / "data" / "bronze" / "crime_data_bronze.parquet")
SILVER_PATH = str(PROJECT_ROOT / "data" / "silver")
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

# Garantir que diretório existe
os.makedirs(SILVER_PATH, exist_ok=True)

print(f"📁 Projeto: {PROJECT_ROOT}")
print(f"📁 Bronze: {BRONZE_PATH}")
print(f"📁 Silver: {SILVER_PATH}")
print(f"🔖 Batch: {BATCH_ID}")

In [ ]:
# Carregar dados Bronze
df_bronze = spark.read.parquet(BRONZE_PATH)
print(f"✅ Dados carregados: {df_bronze.count():,} registros")

In [ ]:
# Transformações: limpeza e feature engineering
# Primeiro, separar colunas de dados e metadados
metadata_cols = [c for c in df_bronze.columns if c.startswith('_')]
data_cols = [c for c in df_bronze.columns if c not in metadata_cols]

df = df_bronze.select(data_cols)

# Conversões de data/hora
df_silver = df \
    .withColumn("date_occurred", F.to_timestamp("DATE OCC", "MM/dd/yyyy hh:mm:ss a")) \
    .withColumn("date_reported", F.to_timestamp("Date Rptd", "MM/dd/yyyy hh:mm:ss a")) \
    .withColumn("hour_occurred", F.substring(F.lpad("TIME OCC", 4, "0"), 1, 2).cast("int")) \
    .withColumn("year_occurred", F.year("date_occurred")) \
    .withColumn("month_occurred", F.month("date_occurred")) \
    .withColumn("day_occurred", F.dayofmonth("date_occurred")) \
    .withColumn("day_of_week", F.dayofweek("date_occurred")) \
    .withColumn("is_weekend", F.when(F.dayofweek("date_occurred").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("period_of_day", 
                F.when(F.col("hour_occurred") < 6, "MADRUGADA")
                 .when(F.col("hour_occurred") < 12, "MANHA")
                 .when(F.col("hour_occurred") < 18, "TARDE")
                 .otherwise("NOITE")) \
    .withColumn("is_violent", 
                F.when(F.col("Crm Cd").isin([110, 113, 121, 122, 210, 220, 230, 231, 235, 236, 250, 251, 310, 320, 510, 520]), 1)
                 .otherwise(0)) \
    .withColumnRenamed("DR_NO", "crime_id") \
    .withColumnRenamed("TIME OCC", "time_occurred") \
    .withColumnRenamed("AREA", "area_code") \
    .withColumnRenamed("AREA NAME", "area_name") \
    .withColumnRenamed("Rpt Dist No", "district_number") \
    .withColumnRenamed("Part 1-2", "part_code") \
    .withColumnRenamed("Crm Cd", "crime_code") \
    .withColumnRenamed("Crm Cd Desc", "crime_description") \
    .withColumnRenamed("Vict Age", "victim_age") \
    .withColumnRenamed("Vict Sex", "victim_sex") \
    .withColumnRenamed("Vict Descent", "victim_descent") \
    .withColumnRenamed("Premis Cd", "premise_code") \
    .withColumnRenamed("Premis Desc", "premise_description") \
    .withColumnRenamed("Weapon Used Cd", "weapon_code") \
    .withColumnRenamed("Weapon Desc", "weapon_description") \
    .withColumnRenamed("Status", "status_code") \
    .withColumnRenamed("Status Desc", "status_description") \
    .withColumnRenamed("LOCATION", "location") \
    .withColumnRenamed("Cross Street", "cross_street") \
    .withColumnRenamed("LAT", "latitude") \
    .withColumnRenamed("LON", "longitude") \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .withColumn("_silver_batch_id", F.lit(BATCH_ID))

print("✅ Transformações aplicadas")

In [ ]:
# Filtrar registros inválidos
df_silver = df_silver \
    .filter(F.col("crime_id").isNotNull()) \
    .filter(F.col("date_occurred").isNotNull()) \
    .filter(F.col("crime_code").isNotNull()) \
    .filter((F.col("latitude") != 0) & (F.col("longitude") != 0))

print(f"✅ Registros após filtros: {df_silver.count():,}")

In [ ]:
# Salvar na camada Silver
df_silver.write \
    .mode("overwrite") \
    .parquet(f"{SILVER_PATH}/crimes.parquet")

print("✅ Dados salvos na camada Silver!")

In [ ]:
# Criar dimensões
dim_areas = df_silver.select("area_code", "area_name").distinct()
dim_areas.write.mode("overwrite").parquet(f"{SILVER_PATH}/dim_areas.parquet")

dim_crime_types = df_silver.select("crime_code", "crime_description", "is_violent").distinct()
dim_crime_types.write.mode("overwrite").parquet(f"{SILVER_PATH}/dim_crime_types.parquet")

dim_weapons = df_silver.select("weapon_code", "weapon_description").distinct().filter(F.col("weapon_code").isNotNull())
dim_weapons.write.mode("overwrite").parquet(f"{SILVER_PATH}/dim_weapons.parquet")

dim_premises = df_silver.select("premise_code", "premise_description").distinct().filter(F.col("premise_code").isNotNull())
dim_premises.write.mode("overwrite").parquet(f"{SILVER_PATH}/dim_premises.parquet")

print("✅ Dimensões salvas!")

In [ ]:
# Finalizar
spark.stop()
print("✅ Job Bronze → Silver concluído!")